# 09 — Análises pós-benchmark (Fase 1, só inferência)

Consolida as três análises antes em `scripts/curva_horizonte.py`, `scripts/diebold_mariano.py` e `scripts/climatologia.py`: curva de erro por horizonte + MASE (Fase 1.1), teste Diebold-Mariano ens × LSTNet (Fase 1.2) e baseline de climatologia (Fase 1.3, descartado).

- Só inferência, com os checkpoints congelados de 02/03 (LSTNet), 04/05 (PatchTST/DLinear) e 06/07 (LGBM, DLinear-residual, pesos do ensemble). 2025 nunca é usado p/ fit/tuning.
- Protocolo idêntico ao 08: grade 5 min, interpolação limite 24 (2 h), janelas L=8640 → H=288, âncoras diárias com alvo às 23:55, janelas com NaN descartadas.
- Saídas (em `resultados/08-benchmark-2025/`): `metricas_por_horizonte_{ph,od}.csv` + `figs/08-mae-por-horizonte-{ph,od}.png`, `metricas_dm_{ph,od}.csv`, `metricas_climatologia_{ph,od}.csv` + `figs/08-climatologia-{ph,od}.png`.
- Exceção de nome: fora do padrão `NN-<modelo>-<variável>` porque não é experimento com treino (não cria pasta própria em `resultados/`).

Reproduzir: `OMP_NUM_THREADS=4 MKL_NUM_THREADS=4 OPENBLAS_NUM_THREADS=4 .venv/bin/jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=2400 notebooks/09-analises-pos-benchmark.ipynb`

In [1]:
import gzip
import json
import math
import os
import pickle
import time
import warnings
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.metrics import mean_absolute_error, mean_squared_error

import torch
import torch.nn as nn

warnings.filterwarnings("ignore")
torch.set_num_threads(4)
torch.set_num_interop_threads(4)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dados" / "treino").exists())
TR = ROOT / "dados" / "treino"
BM = ROOT / "dados" / "benchmark"
OUT = ROOT / "resultados" / "08-benchmark-2025"
(OUT / "figs").mkdir(parents=True, exist_ok=True)

L, H = 8640, 288
SEASON = 288
INTERP_LIMIT = 24
LN, HN = 2016, 288
FILES = {
    "ph": ("ef01-mogi-das-cruzes_ph_2024.csv", "ef01-mogi-das-cruzes_ph_2025.csv", "pH"),
    "od": (
        "ef01-mogi-das-cruzes_oxigenio-dissolvido_2024.csv",
        "ef01-mogi-das-cruzes_oxigenio-dissolvido_2025.csv",
        "Oxigênio Dissolvido (mg/L)",
    ),
}

T0 = time.time()
print("ROOT:", ROOT, "| torch:", torch.__version__)

ROOT: /home/marcos/Projetos/temporal-model | torch: 2.14.0+cpu


## Fase 1.1 — Curva de erro por horizonte + MASE (ex-`scripts/curva_horizonte.py`)

Re-roda os checkpoints sobre as âncoras diárias de 2025 e calcula MAE(h), h=1..288, por modelo + MASE(h) = MAE(h)/denominador, com denominador = média de |y_t − y_{t−288}| na série 2024. Prophet excluído: exigiria refit por origem + CmdStan (não é checkpoint de inferência direta). Trava: o MAE médio do ensemble nas âncoras deve reproduzir os headlines do 08 (pH 0,0509 · OD 0,2107) com 2% — se divergir, aborta sem escrever.

In [2]:
# ---------- arquiteturas (cópias fiéis de app.py / notebook 08) ----------
class LSTNet1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv1d(3, 32, kernel_size=12, stride=6)
        self.gru = nn.GRU(32, 64, batch_first=True)
        self.skipcell = nn.GRUCell(32, 32)
        self.head = nn.Linear(96, 288)
        self.ar = nn.Linear(288, 288)
        self.drop = nn.Dropout(0.1)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))

    def forward(self, xv, tod):
        mu = xv.mean(dim=1, keepdim=True); sg = xv.std(dim=1, keepdim=True).clamp_min(1e-3)
        vn = self.gamma * (xv - mu) / sg + self.beta
        f = self.drop(torch.relu(self.conv(torch.cat([vn.unsqueeze(1), tod.transpose(1, 2)], dim=1))))
        f = f.transpose(1, 2)
        _, h = self.gru(f)
        b, t, _ = f.shape
        hs = torch.zeros(b, 32)
        states = [hs]
        for i in range(t):
            prev = states[i - 48] if i - 48 >= 0 else states[0]
            hs = self.skipcell(f[:, i, :], prev)
            states.append(hs)
        g = self.gamma.clamp_min(1e-3)
        yn = self.head(self.drop(torch.cat([h.squeeze(0), hs], dim=1)))
        ya = self.ar(vn[:, -288:])
        return (yn + ya - self.beta) / g * sg + mu


class PatchTST(nn.Module):
    def __init__(self):
        super().__init__()
        self.N = (LN - 48) // 24 + 1
        self.proj = nn.Linear(48, 64)
        self.pos = nn.Parameter(torch.randn(1, self.N, 64) * 0.02)
        layer = nn.TransformerEncoderLayer(64, 4, 128, 0.1, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, 3)
        self.drop = nn.Dropout(0.1)
        self.head = nn.Linear(self.N * 64, HN)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        mu = x.mean(dim=1, keepdim=True); sg = x.std(dim=1, keepdim=True).clamp_min(1e-3)
        xn = self.gamma * (x - mu) / sg + self.beta
        z = self.proj(xn.unfold(1, 48, 24)) + self.pos
        z = self.enc(self.drop(z))
        y = self.head(self.drop(z.flatten(1)))
        return (y - self.beta) / self.gamma.clamp_min(1e-3) * sg + mu


class DLinearLite(nn.Module):
    def __init__(self, k=25, residual=False):
        super().__init__()
        self.pool = nn.AvgPool1d(k, stride=1, padding=k // 2)
        self.lin_t = nn.Linear(LN, HN)
        self.lin_s = nn.Linear(LN, HN)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
        self.residual = residual

    def forward(self, x):
        mu = x.mean(dim=1, keepdim=True); sg = x.std(dim=1, keepdim=True).clamp_min(1e-3)
        xn = self.gamma * (x - mu) / sg + self.beta
        t = self.pool(xn.unsqueeze(1)).squeeze(1)
        y = self.lin_t(t) + self.lin_s(xn - t)
        yn = (y - self.beta) / self.gamma.clamp_min(1e-3) * sg
        return yn if self.residual else yn + mu


def load_state(cls, rel, **kw):
    m = cls(**kw).to("cpu")
    m.load_state_dict(torch.load(ROOT / "resultados" / rel, map_location="cpu", weights_only=False)["state"])
    return m.eval()


def ler(path, col):
    df = pd.read_csv(path, sep=";", decimal=",", encoding="windows-1252",
                     skiprows=1, parse_dates=["Data hora"], dayfirst=True, na_values=[""])
    df = df.rename(columns={"Data hora": "ds", col: "y"}).sort_values("ds").reset_index(drop=True)
    idx = pd.date_range(df["ds"].min(), df["ds"].max(), freq="5min")
    s_raw = df.set_index("ds")["y"].reindex(idx)
    return s_raw.interpolate(method="time", limit=INTERP_LIMIT)


def mae(a, b):
    return float(mean_absolute_error(a.ravel(), b.ravel()))


def rmse(a, b):
    return float(np.sqrt(mean_squared_error(a.ravel(), b.ravel())))


def base_feats(Xb, E):
    cols = [Xb[:, -k] for k in [1, 2, 3, 6, 12, 24, 36, 72, 144, 287, 288, 289, 576, 2016]]
    phase = np.stack([Xb[:, L - 288 * k] for k in range(1, 8)], axis=1)
    cols += [phase.mean(1), phase.std(1)]
    for w in [12, 36, 144, 288]:
        cols += [Xb[:, -w:].mean(1), Xb[:, -w:].std(1)]
    cols += [Xb[:, -2016:].mean(1)]
    F = np.stack(cols, axis=1)
    em = (E.hour.to_numpy() * 60 + E.minute.to_numpy()).astype(np.float32)
    return F.astype(np.float32), em


def hour_sincos(em, j):
    hh = ((em - (H - 1 - j) * 5) % 1440 // 60).astype(np.float32)
    return np.sin(2 * np.pi * hh / 24).astype(np.float32), np.cos(2 * np.pi * hh / 24).astype(np.float32)


@torch.no_grad()
def fwd1(model, Wln, Tln, rows, tod, batch=512):
    outs = []
    for b in range(0, len(rows), batch):
        xb = torch.from_numpy(Wln[rows[b:b + batch]])
        if tod is None:
            outs.append(model(xb).numpy())
        else:
            outs.append(model(xb, torch.from_numpy(tod[rows[b:b + batch]])).numpy())
    return np.concatenate(outs)


print("defs Fase 1.1 OK")

defs Fase 1.1 OK


In [3]:
CKPTS = {
    "ph": {
        "lstnet": "02-lstnet-ph/modelos/lstnet_ph.pt",
        "patch": "04-patchtst-ph/modelos/patchtst_ph.pt",
        "dlin": "04-patchtst-ph/modelos/dlinear_ph.pt",
        "lgbm": "06-ensemble-ph/modelos/lgbm_steps.pkl.gz",
        "lgbm_fb": "06-ensemble-ph/modelos/lgbm_steps.pkl",
        "dlres": "06-ensemble-ph/modelos/dlinear_res_ph.pt",
        "ens": "06-ensemble-ph/modelos/ensemble.json",
    },
    "od": {
        "lstnet": "03-lstnet-od/modelos/lstnet_od.pt",
        "patch": "05-patchtst-od/modelos/patchtst_od.pt",
        "dlin": "05-patchtst-od/modelos/dlinear_od.pt",
        "lgbm": "07-ensemble-od/modelos/lgbm_steps.pkl.gz",
        "lgbm_fb": "07-ensemble-od/modelos/lgbm_steps.pkl",
        "dlres": "07-ensemble-od/modelos/dlinear_res_od.pt",
        "ens": "07-ensemble-od/modelos/ensemble.json",
    },
}
MODELOS = [
    "persistencia", "sazonal_naive_288", "media_movel_288", "sazonal_lag365",
    "lstnet", "patchtst", "dlinear", "lgbm", "dlres", "ens",
]
TOL_H = 0.02  # validação vs headline do 08
HEADLINE = {"ph": 0.0509, "od": 0.2107}

t0 = time.time()
res_h = {}  # var -> {preds dict, Yd, denom, Ed}
for var, (f24, f25, col) in FILES.items():
    s24 = ler(TR / f24, col)
    s = ler(BM / f25, col)
    # denominador MASE na série 2024 (pares com NaN descartados)
    a24 = s24.to_numpy().astype(float)
    d = np.abs(a24[SEASON:] - a24[:-SEASON])
    denom = float(d[~np.isnan(d)].mean())
    # janelamento idêntico ao 08; origens com NaN descartadas
    v = s.to_numpy().astype(np.float32)
    W = sliding_window_view(v, L + H)
    ok = ~np.isnan(W).any(axis=1)
    W = W[ok]
    X, Y = W[:, :L], W[:, L:]
    ends = s.index[L + H - 1:][ok]
    di = np.where(ends.time == pd.Timestamp("23:55").time())[0]
    Xd, Yd, Ed = X[di], Y[di], ends[di]
    print(f"{var}: {len(X)} janelas | âncoras diárias: {len(di)} "
          f"({Ed[0].date()} → {Ed[-1].date()}) | denom_MASE_2024={denom:.4f}", flush=True)
    # baratos
    preds = {
        "persistencia": np.repeat(Xd[:, -1:], H, axis=1),
        "sazonal_naive_288": np.stack([Xd[:, L - SEASON + h] for h in range(H)], axis=1),
        "media_movel_288": np.repeat(Xd[:, -SEASON:].mean(axis=1, keepdims=True), H, axis=1),
    }
    # lag-365 (mesma data em 2024; fallback honesto p/ saz-288 onde 2024 falha)
    s24v, s24i = s24.to_numpy(), s24.index
    P365 = np.empty_like(Yd)
    fb = 0
    for k in range(len(Yd)):
        e = Ed[k]
        try:
            loc = s24i.get_loc(pd.Timestamp(year=2024, month=e.month, day=e.day,
                                            hour=e.hour, minute=e.minute))
            src = s24v[loc - 287:loc + 1]
        except KeyError:
            src = None
        if src is None or np.isnan(src).any():
            src = Xd[k][L - SEASON:L]
            fb += 1
        P365[k] = src
    preds["sazonal_lag365"] = P365
    print(f"{var}: lag-365 pronto (fallback {fb}/{len(Yd)})", flush=True)
    # redes (só âncoras)
    val5 = v
    SIN5 = np.sin(2 * np.pi * (s.index.hour.to_numpy() * 60 + s.index.minute.to_numpy()) / 1440.0).astype(np.float32)
    COS5 = np.cos(2 * np.pi * (s.index.hour.to_numpy() * 60 + s.index.minute.to_numpy()) / 1440.0).astype(np.float32)
    Wln = sliding_window_view(val5, LN)
    Tln = sliding_window_view(np.stack([SIN5, COS5], axis=1), LN, axis=0).transpose(0, 2, 1).astype(np.float32)
    rowln = s.index.get_indexer(ends - pd.Timedelta(minutes=5 * H)) - LN + 1
    rows = rowln[di]
    C = CKPTS[var]
    for k in ("lstnet", "patch", "dlin", "dlres", "ens"):
        p = ROOT / "resultados" / C[k]
        assert p.exists(), f"checkpoint ausente: {p}"
    lstnet = load_state(LSTNet1D, C["lstnet"])
    patch = load_state(PatchTST, C["patch"])
    dlin = load_state(DLinearLite, C["dlin"])
    dlres = load_state(DLinearLite, C["dlres"], residual=True)
    pk = ROOT / "resultados" / C["lgbm"]
    if not pk.exists():
        pk = ROOT / "resultados" / C["lgbm_fb"]
    assert pk.exists(), f"checkpoint ausente: {pk}"
    opener = gzip.open if pk.suffix == ".gz" else open
    with opener(pk, "rb") as f:
        lgbms = pickle.load(f)
    ens = json.load(open(ROOT / "resultados" / C["ens"]))["pesos"]
    w = [ens["sazonal"], ens["lstnet"], ens["lgbm"], ens["dlres"]]
    Pn = fwd1(lstnet, Wln, Tln, rows, Tln)
    Pt = fwd1(patch, Wln, Tln, rows, None)
    Dl = fwd1(dlin, Wln, Tln, rows, None)
    F, em = base_feats(Xd, Ed)
    S = preds["sazonal_naive_288"]
    Gb = np.empty((len(Xd), H), dtype=np.float32)
    for j, m in enumerate(lgbms):
        sh, ch = hour_sincos(em, j)
        Gb[:, j] = S[:, j] + m.predict(np.column_stack([F, sh, ch]))
    Dr = S + fwd1(dlres, Wln, Tln, rows, None)
    En = w[0] * S + w[1] * Pn + w[2] * Gb + w[3] * Dr
    preds.update({"lstnet": Pn, "patchtst": Pt, "dlinear": Dl, "lgbm": Gb, "dlres": Dr, "ens": En})
    res_h[var] = {"preds": preds, "Yd": Yd, "denom": denom, "Ed": Ed}
    print(f"{var}: inferência OK ({time.time() - t0:.0f}s)", flush=True)

# ---------- validação vs headlines do 08 (aborta se divergir) ----------
for var in ("ph", "od"):
    mae_medio = float(np.abs(res_h[var]["Yd"] - res_h[var]["preds"]["ens"]).mean())
    diff = abs(mae_medio - HEADLINE[var]) / HEADLINE[var]
    print(f"{var}: MAE médio ens (âncoras)={mae_medio:.4f} vs headline {HEADLINE[var]:.4f} "
          f"(Δ={100 * diff:.2f}%, tol={100 * TOL_H:.0f}%)", flush=True)
    if diff > TOL_H:
        raise RuntimeError(f"ABORTO: divergência acima da tolerância em {var} — sem artefatos escritos.")

# ---------- MAE(h) / MASE(h), CSVs e figs ----------
for var in ("ph", "od"):
    Yd, preds, denom = res_h[var]["Yd"], res_h[var]["preds"], res_h[var]["denom"]
    mae_h = {m: np.abs(Yd - p).mean(axis=0) for m, p in preds.items()}
    df = pd.DataFrame({"h": np.arange(1, H + 1)})
    for m in MODELOS:
        df[m] = mae_h[m]
    df["MASE_ens"] = df["ens"] / denom
    df["MASE_lstnet"] = df["lstnet"] / denom
    df["MASE_sazonal"] = df["sazonal_naive_288"] / denom
    df.round(6).to_csv(OUT / f"metricas_por_horizonte_{var}.csv", index=False)

    fig, ax = plt.subplots(figsize=(10, 5.5))
    for m in MODELOS:
        kw = dict(lw=1.1, label=m)
        if m == "ens":
            kw.update(color="red", lw=2.2, zorder=5)
        elif m == "sazonal_lag365":
            kw.update(color="magenta", ls="--", lw=1.0)
        ax.plot(df["h"], df[m], **kw)
    ax.set_yscale("log")
    ax.set_xlabel("horizonte h (passos de 5 min; 288 = 24 h)")
    ax.set_ylabel("MAE (log) — menor é melhor")
    ax.set_title(f"MAE por horizonte — {var} (âncoras diárias 2025, sem Prophet)")
    ax.legend(fontsize=7, ncol=2)
    fig.tight_layout()
    fig.savefig(OUT / "figs" / f"08-mae-por-horizonte-{var}.png")
    plt.close(fig)

    mae_med = {m: float(mae_h[m].mean()) for m in MODELOS}
    mase_med = {k: float((mae_h[{"ens": "ens", "lstnet": "lstnet", "saz": "sazonal_naive_288"}[k]] / denom).mean())
                for k in ("ens", "lstnet", "saz")}
    win = int((mae_h["ens"] <= np.min([mae_h[m] for m in MODELOS if m != "ens"], axis=0)).sum())
    print(f"{var}: MAE médio " + " ".join(f"{m}={mae_med[m]:.4f}" for m in
          ("ens", "lstnet", "sazonal_naive_288")) +
          f" | MASE médio ens={mase_med['ens']:.3f} lstnet={mase_med['lstnet']:.3f} "
          f"saz={mase_med['saz']:.3f} | ens vence em {win}/288 horizontes", flush=True)

print(f"Fase 1.1 OK em {time.time() - t0:.0f}s", flush=True)

ph: 24589 janelas | âncoras diárias: 86 (2025-01-31 → 2025-09-29) | denom_MASE_2024=0.0549


ph: lag-365 pronto (fallback 1/86)


ph: inferência OK (6s)


od: 46556 janelas | âncoras diárias: 163 (2025-01-31 → 2025-12-30) | denom_MASE_2024=0.2199


od: lag-365 pronto (fallback 2/163)


...

ph: MAE médio ens (âncoras)=0.0503 vs headline 0.0509 (Δ=1.14%, tol=2%)


od: MAE médio ens (âncoras)=0.2072 vs headline 0.2107 (Δ=1.67%, tol=2%)


ph: MAE médio ens=0.0503 lstnet=0.0529 sazonal_naive_288=0.0596 | MASE médio ens=0.917 lstnet=0.964 saz=1.086 | ens vence em 208/288 horizontes


od: MAE médio ens=0.2072 lstnet=0.2099 sazonal_naive_288=0.2732 | MASE médio ens=0.942 lstnet=0.955 saz=1.243 | ens vence em 215/288 horizontes


Fase 1.1 OK em 16s


## Fase 1.2 — Diebold-Mariano: ensemble × LSTNet (ex-`scripts/diebold_mariano.py`)

Só ens + LSTNet nas mesmas âncoras diárias de 2025. Métrica por origem: se = erro quadrático médio em H=288. Teste DM sobre d_t = se_ens − se_lstnet (H0: mesma acurácia), erros-padrão HAC/Newey-West com truncagem h = floor(4·(T/100)^(2/9)), bicaudal (p pela normal, sem scipy). Trava: o MAE das âncoras deve reproduzir as réguas do 08 com 2% (pH ens 0,0509/lstnet 0,0529; OD ens 0,2107/lstnet 0,2127) — se divergir, aborta. Séries por origem em `metricas_dm_{ph,od}.csv`.

In [4]:
# Réguas do benchmark rolante 2025 (resultados/08-benchmark-2025/README.md)
REGUAS = {"ph": {"ens": 0.0509, "lstnet": 0.0529}, "od": {"ens": 0.2107, "lstnet": 0.2127}}
CKPT = {
    "ph": {"lstnet": "02-lstnet-ph/modelos/lstnet_ph.pt", "dlres": "06-ensemble-ph/modelos/dlinear_res_ph.pt",
           "lgbm_gz": "06-ensemble-ph/modelos/lgbm_steps.pkl.gz", "lgbm": "06-ensemble-ph/modelos/lgbm_steps.pkl",
           "ens": "06-ensemble-ph/modelos/ensemble.json"},
    "od": {"lstnet": "03-lstnet-od/modelos/lstnet_od.pt", "dlres": "07-ensemble-od/modelos/dlinear_res_od.pt",
           "lgbm_gz": "07-ensemble-od/modelos/lgbm_steps.pkl.gz", "lgbm": "07-ensemble-od/modelos/lgbm_steps.pkl",
           "ens": "07-ensemble-od/modelos/ensemble.json"},
}
TOL_DM = 0.02  # tolerância da trava de validação (2%)


def ler_2025(var: str) -> pd.Series:
    _, fname, col = FILES[var]
    df = pd.read_csv(BM / fname, sep=";", decimal=",", encoding="windows-1252",
                     skiprows=1, parse_dates=["Data hora"], dayfirst=True, na_values=[""])
    df = df.rename(columns={"Data hora": "ds", col: "y"}).sort_values("ds").reset_index(drop=True)
    idx = pd.date_range(df["ds"].min(), df["ds"].max(), freq="5min")
    s = df.set_index("ds")["y"].reindex(idx).interpolate(method="time", limit=INTERP_LIMIT)
    return s


def ancoras_diarias(s: pd.Series):
    v = s.to_numpy().astype(np.float32)
    W = sliding_window_view(v, L + H)
    ok = ~np.isnan(W).any(axis=1)
    W = W[ok]
    ends_all = s.index[L + H - 1:][ok]
    di = np.where(ends_all.time == pd.Timestamp("23:55").time())[0]
    return W[di, :L], W[di, L:], ends_all[di]


def tod_ancoras(fins_ctx: pd.DatetimeIndex) -> np.ndarray:
    """sin/cos hora-do-dia dos últimos LN=2016 slots de cada contexto (como app.py:inferencia)."""
    tod = np.empty((len(fins_ctx), LN, 2), dtype=np.float32)
    for i, fim in enumerate(fins_ctx):
        hh = pd.date_range(fim - pd.Timedelta(minutes=5 * (LN - 1)), periods=LN, freq="5min")
        mnt = (hh.hour.to_numpy() * 60 + hh.minute.to_numpy()).astype(np.float32)
        tod[i, :, 0] = np.sin(2 * np.pi * mnt / 1440.0)
        tod[i, :, 1] = np.cos(2 * np.pi * mnt / 1440.0)
    return tod


@torch.no_grad()
def fwd_dm(modelo, Xb: np.ndarray, tod: np.ndarray | None = None, batch: int = 256) -> np.ndarray:
    outs = []
    for b in range(0, len(Xb), batch):
        xb = torch.from_numpy(Xb[b:b + batch])
        if tod is None:
            outs.append(modelo(xb).numpy())
        else:
            outs.append(modelo(xb, torch.from_numpy(tod[b:b + batch])).numpy())
    return np.concatenate(outs).astype(np.float64)


def inferencia_dm(var: str, Xd: np.ndarray, ends: pd.DatetimeIndex):
    """Prevê ens + lstnet nas âncoras (espelha app.py:inferencia + pesos do ensemble.json)."""
    c = CKPT[var]
    ens = json.load(open(ROOT / "resultados" / c["ens"]))["pesos"]
    w = [ens["sazonal"], ens["lstnet"], ens["lgbm"], ens["dlres"]]
    S = Xd[:, L - SEASON:L].astype(np.float64)  # sazonal-naive lag-288
    fins_ctx = ends - pd.Timedelta(minutes=5 * H)
    Xln = Xd[:, -LN:].astype(np.float32)
    Pn = fwd_dm(load_state(LSTNet1D, c["lstnet"]), Xln, tod_ancoras(fins_ctx))
    Dr = S + fwd_dm(load_state(DLinearLite, c["dlres"], residual=True), Xln)
    if w[2] != 0.0:
        pk_gz, pk = ROOT / "resultados" / c["lgbm_gz"], ROOT / "resultados" / c["lgbm"]
        pk = pk_gz if pk_gz.exists() else pk
        opener = gzip.open if pk.suffix == ".gz" else open
        with opener(pk, "rb") as f:
            lgbms = pickle.load(f)
        F, em = base_feats(Xd.astype(np.float64), ends)
        Gb = np.empty((len(Xd), H), dtype=np.float64)
        for j, mdl in enumerate(lgbms):
            sh, ch = hour_sincos(em, j)
            Gb[:, j] = S[:, j] + mdl.predict(np.column_stack([F, sh, ch]))
        En = w[0] * S + w[1] * Pn + w[2] * Gb + w[3] * Dr
    else:
        En = w[0] * S + w[1] * Pn + w[3] * Dr  # pH: peso lgbm = 0, nem carrega
    return Pn, En


def dm_hac(d: np.ndarray, h: int):
    d = np.asarray(d, dtype=float)
    T = len(d)
    dbar = float(d.mean())
    e = d - dbar
    s = float(e @ e) / T  # gamma_0
    for k in range(1, h + 1):
        gk = float(e[k:] @ e[:T - k]) / T
        s += 2.0 * (1.0 - k / (h + 1.0)) * gk
    var = s / T
    if not np.isfinite(var) or var <= 0:
        return dbar, float("nan"), float("nan")
    dm = dbar / math.sqrt(var)
    p = math.erfc(abs(dm) / math.sqrt(2.0))  # bicaudal, normal
    return dbar, dm, p


t0 = time.time()
res_dm = {}
for var in ["ph", "od"]:
    s = ler_2025(var)
    Xd, Yd, ends = ancoras_diarias(s)
    T = len(Xd)
    print(f"[{var}] âncoras diárias 2025: T={T} ({ends[0].date()} → {ends[-1].date()})", flush=True)
    Pn, En = inferencia_dm(var, Xd, ends)
    se_lst = ((Yd - Pn) ** 2).mean(axis=1)
    se_ens = ((Yd - En) ** 2).mean(axis=1)
    mae_lst = float(np.abs(Yd - Pn).mean())
    mae_ens = float(np.abs(Yd - En).mean())
    # Trava de validação: reproduzir as réguas do benchmark rolante (tol 2%)
    ok = True
    for nome, obt, ref in [("ens", mae_ens, REGUAS[var]["ens"]), ("lstnet", mae_lst, REGUAS[var]["lstnet"])]:
        dev = abs(obt - ref) / ref
        print(f"[{var}] MAE {nome}: obtido={obt:.4f} régua={ref:.4f} desvio={100 * dev:.2f}%", flush=True)
        if dev > TOL_DM:
            ok = False
    gap = 100 * (mae_lst - mae_ens) / mae_lst
    print(f"[{var}] gap ens×lstnet: {gap:.2f}%", flush=True)
    if not ok:
        raise RuntimeError(f"[{var}] VALIDAÇÃO FALHOU (desvio > 2%) — PARE. Verifique janelamento/checkpoints.")
    d = se_ens - se_lst
    h = math.floor(4 * (T / 100) ** (2 / 9))
    dbar, dm, p = dm_hac(d, h)
    win_ens = 100 * float((se_ens < se_lst).mean())
    win_lst = 100 * float((se_lst < se_ens).mean())
    print(f"[{var}] DM: dbar={dbar:.3e} h={h} T={T} stat={dm:.3f} p={p:.3g} | "
          f"vitórias ens={win_ens:.1f}% lstnet={win_lst:.1f}%", flush=True)
    pd.DataFrame({"origem": ends.strftime("%Y-%m-%d %H:%M:%S"),
                  "se_ens": se_ens, "se_lstnet": se_lst}).to_csv(
        OUT / f"metricas_dm_{var}.csv", index=False)
    print(f"[{var}] salvo: metricas_dm_{var}.csv ({T} origens)", flush=True)
    res_dm[var] = dict(T=T, h=h, mae_ens=mae_ens, mae_lst=mae_lst, gap=gap,
                       dbar=dbar, dm=dm, p=p, win_ens=win_ens, win_lst=win_lst,
                       reg_ens=REGUAS[var]["ens"], reg_lst=REGUAS[var]["lstnet"])

for var, nome in [("ph", "pH"), ("od", "OD")]:
    r = res_dm[var]
    if r["p"] < 0.05 and r["dbar"] < 0:
        verd = "o ganho do ensemble sobre o LSTNet é estatisticamente significativo (H0 rejeitada a 5%)"
    elif r["p"] < 0.05 and r["dbar"] > 0:
        verd = "o LSTNet supera o ensemble de forma estatisticamente significativa (H0 rejeitada a 5%)"
    else:
        verd = "não se rejeita H0 de mesma acurácia a 5%: o gap médio é indistinguível de ruído amostral"
    print(f"{nome} (T={r['T']}, h={r['h']}): DM={r['dm']:.3f}, p={r['p']:.3g} — {verd}", flush=True)

print(f"Fase 1.2 OK em {time.time() - t0:.0f}s", flush=True)

[ph] âncoras diárias 2025: T=86 (2025-01-31 → 2025-09-29)


[ph] MAE ens: obtido=0.0503 régua=0.0509 desvio=1.14%


[ph] MAE lstnet: obtido=0.0529 régua=0.0529 desvio=0.05%


[ph] gap ens×lstnet: 4.92%


[ph] DM: dbar=-4.994e-04 h=3 T=86 stat=-2.876 p=0.00403 | vitórias ens=77.9% lstnet=22.1%


...

[od] DM: dbar=1.534e-03 h=4 T=163 stat=0.732 p=0.464 | vitórias ens=58.3% lstnet=41.7%


[od] salvo: metricas_dm_od.csv (163 origens)


pH (T=86, h=3): DM=-2.876, p=0.00403 — o ganho do ensemble sobre o LSTNet é estatisticamente significativo (H0 rejeitada a 5%)


OD (T=163, h=4): DM=0.732, p=0.464 — não se rejeita H0 de mesma acurácia a 5%: o gap médio é indistinguível de ruído amostral


Fase 1.2 OK em 8s


## Fase 1.3 — Climatologia, baseline de custo zero, DESCARTADO (ex-`scripts/climatologia.py`)

Calibração SOMENTE em 2024: p/ cada slot de 5 min do dia (288) e cada dia do calendário, o valor médio daquela época, suavizado com média móvel centrada de ~15 dias sobre o dia-do-ano (circular, nan-aware, min_periods=1). Avaliação nas mesmas âncoras 23:55 de 2025 (L/H só reproduzem as âncoras — irrelevantes p/ a previsão, que é o perfil da época). Nenhum parâmetro tocado em 2025. Saídas: `metricas_climatologia_{ph,od}.csv` + `figs/08-climatologia-{ph,od}.png`.

In [5]:
SMOOTH_DAYS = 15  # média móvel ~15 dias sobre o dia-do-ano
# Réguas do 08 (benchmark 2025, rolante) — só p/ impressão comparativa, sem re-fit.
REGUAS_CLIMA = {"ph": {"saz": 0.0597, "ens": 0.0509}, "od": {"saz": 0.2714, "ens": 0.2107}}


def constroi_climatologia(s24):
    """Perfil (dia-do-calendario x slot 288) a partir SÓ de 2024.

    Cada célula (dia, slot) recebe a média dos valores 2024 daquele dia/slot
    (1 obs por dia em 1 ano) e cada coluna-slot é suavizada com média móvel
    centrada de SMOOTH_DAYS dias, circular (dez<->jan) e nan-aware
    (min_periods=1: dias de outage usam os vizinhos disponíveis).
    Células ainda NaN após suavizar (miolo do outage de ~17 d do pH) usam o
    fallback honesto da média anual 2024 daquele slot (ciclo diário médio).
    """
    dias = s24.index.normalize().unique().sort_values()
    n_dias = len(dias)
    slot = (s24.index.hour * 12 + s24.index.minute // 5).to_numpy()
    dia_pos = s24.index.normalize().map(pd.Series(np.arange(n_dias), index=dias)).to_numpy()
    m = np.full((n_dias, 288), np.nan)
    v = s24.to_numpy()
    ok = ~np.isnan(v)
    # média por (dia, slot): em regra 1 obs por célula; mean por segurança
    soma = np.zeros((n_dias, 288))
    conta = np.zeros((n_dias, 288))
    np.add.at(soma, (dia_pos[ok], slot[ok]), v[ok])
    np.add.at(conta, (dia_pos[ok], slot[ok]), 1)
    with np.errstate(invalid="ignore"):
        m = soma / np.maximum(conta, 1)
    m[conta == 0] = np.nan

    pad = SMOOTH_DAYS // 2
    mp = np.concatenate([m[-pad:], m, m[:pad]], axis=0)
    clim = (
        pd.DataFrame(mp)
        .rolling(SMOOTH_DAYS, center=True, min_periods=1)
        .mean()
        .to_numpy()[pad:pad + n_dias]
    )
    # Fallback 2024-only p/ miolos de outage maiores que a janela de suavização.
    slot_mean = np.nanmean(np.where(conta > 0, soma / np.maximum(conta, 1), np.nan), axis=0)
    fb = int(np.isnan(clim).sum())
    clim = np.where(np.isnan(clim), slot_mean[None, :], clim)
    lookup = {(d.month, d.day): i for i, d in enumerate(dias)}
    return clim, lookup, fb, n_dias


def ancoras_clima(s25):
    """Mesmas âncoras 23:55 do 08: fim de janela L+H limpa de NaN (via cumsum, O(n)).

    Equivalente à máscara `~isnan(W).any()` do sliding_window_view do 08.
    Retorna (Yd [n_âncoras x 288], S [saz-288 p/ a figura], rótulos de data).
    """
    v = s25.to_numpy().astype(np.float64)
    bad = np.isnan(v).astype(np.int64)
    c = np.concatenate([[0], np.cumsum(bad)])
    wsum = c[L + H:] - c[: len(v) - (L + H) + 1]
    ok = wsum == 0
    idx = np.arange(L + H - 1, len(v))
    di = [i for i in idx if ok[i - (L + H - 1)] and s25.index[i].time() == pd.Timestamp("23:55").time()]
    di = np.array(di)
    Yd = np.stack([v[i - H + 1: i + 1] for i in di])
    S = np.stack([v[i - H - 288 + 1: i - H + 1] for i in di])  # sazonal-naive-288
    rotulos = [str(s25.index[i].date()) for i in di]
    return Yd, S, rotulos, di


t0 = time.time()
for var, (f24, f25, col) in FILES.items():
    s24 = ler(TR / f24, col)  # calibração: SÓ 2024
    s25 = ler(BM / f25, col)  # avaliação: SÓ 2025 (nunca na calibração)
    clim, lookup, fb, n_dias = constroi_climatologia(s24)
    Yd, S, rotulos, di = ancoras_clima(s25)
    P = np.stack([clim[lookup[(s25.index[i].month, s25.index[i].day)]] for i in di])

    print(f"== {var}: dias-calendário 2024={n_dias} | fallback ciclo-diário={fb} células | âncoras 2025={len(di)} "
          f"({rotulos[0]} -> {rotulos[-1]})")
    # Checagem de paridade das âncoras: o sazonal recalculado aqui deve bater o 08.
    print(f"   saz-288 recalculado: MAE={mae(Yd, S):.4f} RMSE={rmse(Yd, S):.4f} "
          f"(régua 08 rolante: {REGUAS_CLIMA[var]['saz']:.4f})")
    print(f"   CLIMATOLOGIA: MAE={mae(Yd, P):.4f} RMSE={rmse(Yd, P):.4f} "
          f"| réguas 08: saz={REGUAS_CLIMA[var]['saz']:.4f} ens={REGUAS_CLIMA[var]['ens']:.4f}")

    tab = pd.DataFrame(
        {"origem": rotulos,
         "mae": [mae(Yd[k:k + 1], P[k:k + 1]) for k in range(len(di))],
         "rmse": [rmse(Yd[k:k + 1], P[k:k + 1]) for k in range(len(di))]}
    ).round(4)
    tab.to_csv(OUT / f"metricas_climatologia_{var}.csv", index=False)

    ks = [0, len(di) // 2]
    fig, axes = plt.subplots(2, 1, figsize=(12, 6))
    for ax, k in zip(axes, ks):
        tf = pd.date_range(s25.index[di[k]] - pd.Timedelta(minutes=5 * (H - 1)), s25.index[di[k]], freq="5min")
        ax.plot(tf, Yd[k], "k-", lw=1.2, label="real 2025")
        ax.plot(tf, P[k], lw=1, label=f"climatologia (MAE {mae(Yd[k:k+1], P[k:k+1]):.4f})")
        ax.plot(tf, S[k], ":", lw=1, label=f"saz-288 (MAE {mae(Yd[k:k+1], S[k:k+1]):.4f})")
        ax.set_title(f"{var} {rotulos[k]} — real x climatologia (calibrada só em 2024) x sazonal")
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "figs" / f"08-climatologia-{var}.png")
    plt.close(fig)
    print(f"   salvos: metricas_climatologia_{var}.csv + figs/08-climatologia-{var}.png")

print(f"Fase 1.3 OK em {time.time() - t0:.0f}s", flush=True)

== ph: dias-calendário 2024=366 | fallback ciclo-diário=915 células | âncoras 2025=86 (2025-01-31 -> 2025-09-29)
   saz-288 recalculado: MAE=0.0596 RMSE=0.0821 (régua 08 rolante: 0.0597)
   CLIMATOLOGIA: MAE=0.4627 RMSE=0.5330 | réguas 08: saz=0.0597 ens=0.0509


   salvos: metricas_climatologia_ph.csv + figs/08-climatologia-ph.png


== od: dias-calendário 2024=366 | fallback ciclo-diário=0 células | âncoras 2025=163 (2025-01-31 -> 2025-12-30)
   saz-288 recalculado: MAE=0.2732 RMSE=0.4043 (régua 08 rolante: 0.2714)
   CLIMATOLOGIA: MAE=0.8678 RMSE=1.1161 | réguas 08: saz=0.2714 ens=0.2107


   salvos: metricas_climatologia_od.csv + figs/08-climatologia-od.png
Fase 1.3 OK em 7s


## Verificação — artefatos escritos (10 arquivos, mesmos nomes dos scripts)

In [6]:
print(f"TEMPO TOTAL: {time.time() - T0:.0f}s")
for f in ["metricas_por_horizonte_ph.csv", "metricas_por_horizonte_od.csv",
          "metricas_dm_ph.csv", "metricas_dm_od.csv",
          "metricas_climatologia_ph.csv", "metricas_climatologia_od.csv",
          "figs/08-mae-por-horizonte-ph.png", "figs/08-mae-por-horizonte-od.png",
          "figs/08-climatologia-ph.png", "figs/08-climatologia-od.png"]:
    p = OUT / f
    assert p.exists(), f"artefato ausente: {p}"
    print(f"{f}: {p.stat().st_size} bytes")

TEMPO TOTAL: 31s
metricas_por_horizonte_ph.csv: 34425 bytes
metricas_por_horizonte_od.csv: 34424 bytes
metricas_dm_ph.csv: 5422 bytes
metricas_dm_od.csv: 9874 bytes
metricas_climatologia_ph.csv: 2147 bytes
metricas_climatologia_od.csv: 4061 bytes
figs/08-mae-por-horizonte-ph.png: 179434 bytes
figs/08-mae-por-horizonte-od.png: 103883 bytes
figs/08-climatologia-ph.png: 198738 bytes
figs/08-climatologia-od.png: 88741 bytes
